# The Algebra of Tuple Morphisms

CuTe equips layouts with an algebra: composition, coalescence, complements, logical division and logical product. The paper *Categorical Foundations for CuTe Layouts* (Colfax Research) lifts each of these operations to the category $\text{Tuple}$, where they become transparent combinatorial constructions on tuple morphisms, and proves that the translation $f \mapsto L_f$ intertwines the two algebras.

This notebook demonstrates each operation of `tract` on a small explicit example and **cross-checks it against pycute**, NVIDIA's pure-Python reference implementation of CuTe — mirroring the predicates in `tract`'s cross-validation test suite.

## Setup and conventions

Two conventions to keep in mind throughout:

* **Composition order.** For layouts, $B \circ A$ means "apply $A$, then $B$". For morphisms, `tract` writes `f.compose(g)` for "$f$ then $g$", i.e. $g \circ f$ — so the layout cross-check of `f.compose(g)` is `composition(L_g, L_f)`.
* **Comparison up to trivial modes.** Layouts that differ only in size-1 modes or in mode grouping define the same function; the helper `layouts_agree` compares flat modes after nullifying strides of size-1 modes, and `flatten_layout` discards mode grouping. Some identities hold only up to coalescence, and we say so where it matters.

In [1]:
from pycute import Layout, size, complement, logical_divide

from tract import TupleMorphism
from tract.backends import pycute as backend

def L(f):
    """The flat layout of a tuple morphism, via the pycute backend."""
    return backend.compute_flat_layout(f)

## Composition

Tuple morphisms $f: S \to T$ and $g: T \to U$ compose whenever the codomain of $f$ equals the domain of $g$: the underlying pointed maps compose, with the basepoint absorbing everything mapped to it. In `tract`, `f.compose(g)` computes $g \circ f$.

In [2]:
f = TupleMorphism(domain=(2, 2, 2, 2), codomain=(2, 2, 2, 2, 2, 2), map=(3, 2, 6, 5))
g = TupleMorphism(domain=(2, 2, 2, 2, 2, 2), codomain=(2, 2, 2, 2), map=(1, 0, 2, 0, 3, 4))
g_of_f = f.compose(g)
print("f     =", f)
print("g     =", g)
print("g . f =", g_of_f)

f     = (2, 2, 2, 2) --(3, 2, 6, 5)--> (2, 2, 2, 2, 2, 2)
g     = (2, 2, 2, 2, 2, 2) --(1, 0, 2, 0, 3, 4)--> (2, 2, 2, 2)
g . f = (2, 2, 2, 2) --(2, 0, 4, 3)--> (2, 2, 2, 2)


Note how basepoints propagate: domain mode 2 of $f$ maps to codomain mode 2, which $g$ collapses to the basepoint, so mode 2 of the composite is a basepoint.

**Cross-check.** Composition of morphisms matches CuTe composition of layouts:

$$L_{g \circ f} = L_g \circ L_f.$$

Since `f.compose(g)` is "$f$ then $g$", the pycute call is `composition(L_g, L_f)` (wrapped as `backend.compose_layouts`, which handles a pycute rank-0 edge case).

In [3]:
L_f, L_g = L(f), L(g)
layout_of_composite = L(g_of_f)
composite_of_layouts = backend.compose_layouts(L_g, L_f)
print("L_f           =", L_f)
print("L_g           =", L_g)
print("L_(g.f)       =", layout_of_composite)
print("L_g . L_f     =", composite_of_layouts)
print("agree:", backend.layouts_agree(layout_of_composite, composite_of_layouts))

L_f           = (2, 2, 2, 2):(4, 2, 32, 16)
L_g           = (2, 2, 2, 2, 2, 2):(1, 0, 2, 0, 4, 8)
L_(g.f)       = (2, 2, 2, 2):(2, 0, 8, 4)
L_g . L_f     = (2, 2, 2, 2):(2, 0, 8, 4)
agree: True


Non-composable morphisms are rejected — no silent mutual refinement happens at this level (that is the subject of *weak composition* in the nested theory):

In [4]:
try:
    f.compose(f)   # codomain of f is (2,2,2,2,2,2), domain is (2,2,2,2)
except ValueError as e:
    print("ValueError:", e)

ValueError: The given morphisms are not composable.


## Coalescence

Coalescence reduces a morphism to a minimal-complexity form: size-1 modes are dropped, and *adjacent* domain modes that map to *consecutive* codomain modes are merged (their sizes multiply), as are the corresponding codomain modes. `f.coalesce()` computes it.

In [5]:
f = TupleMorphism(domain=(2, 2, 10, 10), codomain=(2, 2, 2, 10, 10), map=(1, 2, 4, 5))
coal_f = f.coalesce()
print("f       =", f)
print("coal(f) =", coal_f)

f       = (2, 2, 10, 10) --(1, 2, 4, 5)--> (2, 2, 2, 10, 10)
coal(f) = (4, 100) --(1, 3)--> (4, 2, 100)


Domain modes 1, 2 map to consecutive codomain modes 1, 2 and merge to a single mode of size $4$; modes 3, 4 similarly merge to $100$. Codomain mode 3 is not hit and blocks further merging — it survives as the gap in the coalesced codomain $(4, 2, 100)$.

**Cross-check.** Coalescence of morphisms matches CuTe coalescence of layouts:

$$L_{\operatorname{coal}(f)} = \operatorname{coal}(L_f).$$

In [6]:
layout_of_coalesce = L(coal_f)
coalesce_of_layout = backend.coalesce_layout(L(f))
print("L_f          =", L(f))
print("L_(coal f)   =", layout_of_coalesce)
print("coal(L_f)    =", coalesce_of_layout)
print("agree:", backend.layouts_agree(layout_of_coalesce, coalesce_of_layout))

L_f          = (2, 2, 10, 10):(1, 2, 8, 80)
L_(coal f)   = (4, 100):(1, 8)
coal(L_f)    = (4, 100):(1, 8)
agree: True


## Sorting

A morphism is **sorted** when its basepoint modes come first (ordered by size) and the remaining modes appear in the order of their images. `f.sort()` precomposes with the permutation that achieves this — the analogue, on morphisms, of sorting the modes of a layout by stride.

In [7]:
f = TupleMorphism(domain=(4, 2, 8), codomain=(2, 4, 2, 8), map=(2, 0, 4))
sorted_f = f.sort()
print("f        =", f, "   sorted:", f.is_sorted())
print("sort(f)  =", sorted_f, "   sorted:", sorted_f.is_sorted())

f        = (4, 2, 8) --(2, 0, 4)--> (2, 4, 2, 8)    sorted: False
sort(f)  = (2, 4, 8) --(0, 2, 4)--> (2, 4, 2, 8)    sorted: True


The basepoint mode (size 2) moves to the front, and the remaining modes follow their images. On the layout side this is exactly sorting the flat modes by stride (breaking ties by shape), available as `backend.sort_flat_layout`:

In [8]:
layout_of_sorted = L(sorted_f)
sorted_layout = backend.sort_flat_layout(L(f))
print("L_f            =", L(f))
print("L_(sort f)     =", layout_of_sorted)
print("sort(L_f)      =", sorted_layout)
print("agree:", layout_of_sorted == sorted_layout)

L_f            = (4, 2, 8):(2, 0, 16)
L_(sort f)     = (2, 4, 8):(0, 2, 16)
sort(L_f)      = (2, 4, 8):(0, 2, 16)
agree: True


## Complement

A morphism $f: S \to T$ is **complementable** when its map has no basepoint entries. Its complement $f^c$ has as domain the codomain modes *not hit* by $f$, included into $T$ in order — so that the concatenation of $f$ and $f^c$ (see below) is an isomorphism onto $T$. `f.complement()` computes it.

In [9]:
f = TupleMorphism(domain=(2, 2), codomain=(2, 5, 2, 5), map=(1, 3))
comp_f = f.complement()
print("f   =", f)
print("f^c =", comp_f)

f   = (2, 2) --(1, 3)--> (2, 5, 2, 5)
f^c = (5, 5) --(2, 4)--> (2, 5, 2, 5)


**Cross-check.** The layout of $f^c$ matches the CuTe complement $\operatorname{comp}(L_f, N)$ with respect to the ambient size $N = \operatorname{cosize}(f)$, **up to coalescence** — CuTe's complement is aggressively coalesced, while $f^c$ retains the mode structure of the codomain:

$$\operatorname{coal}\big(L_{f^c}\big) = \operatorname{coal}\big(\operatorname{comp}(L_f,\ \operatorname{cosize} f)\big).$$

In [10]:
layout_of_complement = L(comp_f)
complement_of_layout = complement(L(f), f.cosize())
print("L_f              =", L(f))
print("L_(f^c)          =", layout_of_complement)
print("comp(L_f, N)     =", complement_of_layout, "   with N =", f.cosize())
print("agree up to coalescence:",
      backend.layouts_agree(backend.coalesce_layout(layout_of_complement),
                            backend.coalesce_layout(complement_of_layout)))

L_f              = (2, 2):(1, 10)
L_(f^c)          = (5, 5):(2, 20)
comp(L_f, N)     = (5, 5):(2, 20)    with N = 100
agree up to coalescence: True


## Concatenation

Morphisms $f, g$ with the **same codomain** and **disjoint images** concatenate: `f.concat(g)` places the domains side by side over the union of the two maps. This is the morphism-level version of stacking layouts.

In [11]:
f = TupleMorphism(domain=(4, 4), codomain=(4, 2, 4), map=(1, 3))
g = TupleMorphism(domain=(2,), codomain=(4, 2, 4), map=(2,))
concat_fg = f.concat(g)
print("f            =", f)
print("g            =", g)
print("concat(f, g) =", concat_fg)

f            = (4, 4) --(1, 3)--> (4, 2, 4)
g            = (2,) --(2,)--> (4, 2, 4)
concat(f, g) = (4, 4, 2) --(1, 3, 2)--> (4, 2, 4)


Here $g$ fills exactly the codomain mode that $f$ skips, so the concatenation is an isomorphism — in fact $g = f^c$, and `f.is_complementary_to(g)` confirms it.

**Cross-check.** Concatenation of morphisms matches flat concatenation of layouts *on the nose* (no flattening or coalescence needed):

In [12]:
print("f.is_complementary_to(g):", f.is_complementary_to(g))
layout_of_concat = L(concat_fg)
concat_of_layouts = backend.flat_concatenate(L(f), L(g))
print("L_(concat)         =", layout_of_concat)
print("concat(L_f, L_g)   =", concat_of_layouts)
print("equal:", layout_of_concat == concat_of_layouts)

f.is_complementary_to(g): True
L_(concat)         = (4, 4, 2):(1, 8, 4)
concat(L_f, L_g)   = (4, 4, 2):(1, 8, 4)
equal: True


Concatenation with overlapping images is rejected:

In [13]:
try:
    f.concat(f)
except ValueError as e:
    print("ValueError:", e)

ValueError: Morphisms must have disjoint images.


## Flat divide

For a complementable $g$ whose codomain equals the domain of $f$, the **flat division** of $f$ by $g$ is

$$f \oslash g \;=\; f \circ \operatorname{concat}(g,\ g^c),$$

computed by `f.flat_divide(g)`. It reorganizes the domain of $f$ into the part selected by $g$ (the "tile") followed by everything else (the "rest") — the flat version of CuTe's `logical_divide`.

In [14]:
f = TupleMorphism(domain=(4, 8, 4, 8), codomain=(4, 8, 4, 8), map=(1, 2, 3, 4))
g = TupleMorphism(domain=(4, 4), codomain=(4, 8, 4, 8), map=(1, 3))
quotient = f.flat_divide(g)
print("f       =", f)
print("g       =", g)
print("f / g   =", quotient)

f       = (4, 8, 4, 8) --(1, 2, 3, 4)--> (4, 8, 4, 8)
g       = (4, 4) --(1, 3)--> (4, 8, 4, 8)
f / g   = (4, 4, 8, 8) --(1, 3, 2, 4)--> (4, 8, 4, 8)


**Cross-check.** The layout of $f \oslash g$ matches `pycute.logical_divide`$(L_f, L_g)$ after flattening (logical divide groups its result into tile/rest modes) and up to coalescence:

In [15]:
layout_of_quotient = L(quotient)
divide_of_layouts = backend.flatten_layout(logical_divide(L(f), L(g)))
print("L_f                        =", L(f))
print("L_g                        =", L(g))
print("L_(f/g)                    =", layout_of_quotient)
print("logical_divide(L_f, L_g)^b =", divide_of_layouts)
print("agree up to coalescence:",
      backend.layouts_agree(backend.coalesce_layout(layout_of_quotient),
                            backend.coalesce_layout(divide_of_layouts)))

L_f                        = (4, 8, 4, 8):(1, 4, 32, 128)
L_g                        = (4, 4):(1, 32)
L_(f/g)                    = (4, 4, 8, 8):(1, 32, 4, 128)
logical_divide(L_f, L_g)^b = (4, 4, 8, 8):(1, 32, 4, 128)
agree up to coalescence: True


## Flat product

For a complementable $f$ and a morphism $g$ whose codomain equals the domain of $f^c$, the **flat product** is

$$f \otimes g \;=\; \operatorname{concat}\!\big(f,\ g \circ f^c\big),$$

computed by `f.flat_product(g)`. The first factor lays out one tile; the second factor, threaded through the complement, describes how the tile is repeated across the remaining space — the flat version of CuTe's `logical_product`.

In [16]:
f = TupleMorphism(domain=(2, 2), codomain=(2, 2, 5, 5), map=(1, 2))
g = TupleMorphism(domain=(5, 5), codomain=(5, 5), map=(2, 1))
product = f.flat_product(g)
print("f       =", f)
print("g       =", g)
print("f (x) g =", product)

f       = (2, 2) --(1, 2)--> (2, 2, 5, 5)
g       = (5, 5) --(2, 1)--> (5, 5)
f (x) g = (2, 2, 5, 5) --(1, 2, 4, 3)--> (2, 2, 5, 5)


**Cross-check.** The layout of $f \otimes g$ equals the flattening of `logical_product`$(L_f, L_g)$ exactly (using `backend.logical_product_layouts`, which wraps pycute's `logical_product` around a rank-0 edge case):

In [17]:
layout_of_product = L(product)
product_of_layouts = backend.flatten_layout(
    backend.logical_product_layouts(L(f), L(g))
)
print("L_f                          =", L(f))
print("L_g                          =", L(g))
print("L_(f x g)                    =", layout_of_product)
print("logical_product(L_f, L_g)^b  =", product_of_layouts)
print("equal:", layout_of_product == product_of_layouts)

L_f                          = (2, 2):(1, 2)
L_g                          = (5, 5):(5, 1)
L_(f x g)                    = (2, 2, 5, 5):(1, 2, 20, 4)
logical_product(L_f, L_g)^b  = (2, 2, 5, 5):(1, 2, 20, 4)
equal: True


## Summary

Every operation of the CuTe layout algebra has a combinatorial avatar on tuple morphisms, and $f \mapsto L_f$ intertwines the two:

| `tract` operation | pycute cross-check | agreement |
|---|---|---|
| `f.compose(g)` ("f then g") | `composition(L_g, L_f)` | up to trivial modes |
| `f.coalesce()` | `coalesce(L_f)` | up to trivial modes |
| `f.sort()` | sort flat modes by stride | exact |
| `f.complement()` | `complement(L_f, f.cosize())` | up to coalescence |
| `f.concat(g)` | flat concatenation | exact |
| `f.flat_divide(g)` | `logical_divide(L_f, L_g)` flattened | up to coalescence |
| `f.flat_product(g)` | `logical_product(L_f, L_g)` flattened | exact |

These are precisely the predicates verified — over hundreds of randomized morphisms — in `tract`'s cross-validation test suite (`tests/test_cross_validation.py`). The same checks also run against a second backend, the cutlass **CuTe DSL** (`tract.backends.cute_dsl`, installed via `pip install "tract[cute]"`), which validates the operations against NVIDIA's production implementation rather than the pycute reference.